# Comprehensive Subject-Level Accuracy Analysis

**Purpose**: Identify subjects dragging down overall model accuracy and examine their performance patterns across different trial types.

**Analysis Strategy**:
1. Overall subject performance (all trials)
2. Performance by ambiguity group (Low, Medium, High)
3. Performance by reaction time group (Fast, Medium, Slow)
4. Cross-cutting patterns: Which subjects consistently underperform?
5. Trial-level investigation for lowest performers

In [9]:
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from src.visualization.plots import set_style

np.random.seed(42)
set_style('whitegrid')

print("="*80)
print("COMPREHENSIVE SUBJECT-LEVEL ACCURACY ANALYSIS")
print("="*80)

COMPREHENSIVE SUBJECT-LEVEL ACCURACY ANALYSIS


## 1. Load All Subject Accuracy Data

In [10]:
# Paths
PRE_DIR = Path('../../data/results/fusion_model_results_PRE')
OUTPUT_DIR = Path('../../data/results/analysis_outputs_PRE')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load overall model accuracies
overall_acc = pd.read_csv(PRE_DIR / 'late_fusion_model_PRE_subject_accuracies.csv')
overall_acc['analysis_type'] = 'Overall'

print(f"\nLoaded overall accuracies: {len(overall_acc)} subjects")
print(f"Mean accuracy: {overall_acc['accuracy'].mean():.4f}")
print(f"SD: {overall_acc['accuracy'].std():.4f}")
print(f"Range: [{overall_acc['accuracy'].min():.4f}, {overall_acc['accuracy'].max():.4f}]")


Loaded overall accuracies: 97 subjects
Mean accuracy: 0.6931
SD: 0.1313
Range: [0.3306, 0.9848]


In [11]:
# Load ambiguity group accuracies
amb_low = pd.read_csv(PRE_DIR / 'ambiguity_group_late_fusion_PRE_Low_subject_accuracies.csv')
amb_low['ambiguity_group'] = 'Low'

amb_medium = pd.read_csv(PRE_DIR / 'ambiguity_group_late_fusion_PRE_Medium_subject_accuracies.csv')
amb_medium['ambiguity_group'] = 'Medium'

amb_high = pd.read_csv(PRE_DIR / 'ambiguity_group_late_fusion_PRE_High_subject_accuracies.csv')
amb_high['ambiguity_group'] = 'High'

# Combine ambiguity groups
ambiguity_acc = pd.concat([amb_low, amb_medium, amb_high], ignore_index=True)

print(f"\nLoaded ambiguity group accuracies:")
print(f"  Low: {len(amb_low)} subjects, mean={amb_low['accuracy'].mean():.4f}")
print(f"  Medium: {len(amb_medium)} subjects, mean={amb_medium['accuracy'].mean():.4f}")
print(f"  High: {len(amb_high)} subjects, mean={amb_high['accuracy'].mean():.4f}")


Loaded ambiguity group accuracies:
  Low: 97 subjects, mean=0.8070
  Medium: 97 subjects, mean=0.6860
  High: 97 subjects, mean=0.6075


In [12]:
# Load reaction time group accuracies
rt_fast = pd.read_csv(PRE_DIR / 'reaction_time_group_late_fusion_PRE_Fast_subject_accuracies.csv')
rt_fast['rt_group'] = 'Fast'

rt_slow = pd.read_csv(PRE_DIR / 'reaction_time_group_late_fusion_PRE_Slow_subject_accuracies.csv')
rt_slow['rt_group'] = 'Slow'

# Combine RT groups
rt_acc = pd.concat([rt_fast, rt_slow], ignore_index=True)

print(f"\nLoaded reaction time group accuracies:")
print(f"  Fast: {len(rt_fast)} subjects, mean={rt_fast['accuracy'].mean():.4f}")
print(f"  Slow: {len(rt_slow)} subjects, mean={rt_slow['accuracy'].mean():.4f}")


Loaded reaction time group accuracies:
  Fast: 97 subjects, mean=0.7223
  Slow: 97 subjects, mean=0.6361


## 2. Overall Subject Performance Analysis

In [13]:
print("\n" + "="*80)
print("OVERALL SUBJECT PERFORMANCE (ALL TRIALS)")
print("="*80)

# Descriptive statistics
print(f"\nDescriptive Statistics:")
print(f"  N subjects: {len(overall_acc)}")
print(f"  Mean: {overall_acc['accuracy'].mean():.4f}")
print(f"  Median: {overall_acc['accuracy'].median():.4f}")
print(f"  SD: {overall_acc['accuracy'].std():.4f}")
print(f"  SEM: {stats.sem(overall_acc['accuracy']):.4f}")
print(f"  Min: {overall_acc['accuracy'].min():.4f}")
print(f"  Max: {overall_acc['accuracy'].max():.4f}")
print(f"  Range: {overall_acc['accuracy'].max() - overall_acc['accuracy'].min():.4f}")

# Percentiles
print(f"\nPercentiles:")
for p in [10, 25, 50, 75, 90]:
    print(f"  {p}th: {overall_acc['accuracy'].quantile(p/100):.4f}")

# Performance categories
below_chance = overall_acc[overall_acc['accuracy'] < 0.5]
below_60 = overall_acc[overall_acc['accuracy'] < 0.6]
above_80 = overall_acc[overall_acc['accuracy'] >= 0.8]
above_90 = overall_acc[overall_acc['accuracy'] >= 0.9]

print(f"\nPerformance Categories:")
print(f"  Below chance (<50%): {len(below_chance)} ({100*len(below_chance)/len(overall_acc):.1f}%)")
print(f"  Below 60%: {len(below_60)} ({100*len(below_60)/len(overall_acc):.1f}%)")
print(f"  Above 80%: {len(above_80)} ({100*len(above_80)/len(overall_acc):.1f}%)")
print(f"  Above 90%: {len(above_90)} ({100*len(above_90)/len(overall_acc):.1f}%)")


OVERALL SUBJECT PERFORMANCE (ALL TRIALS)

Descriptive Statistics:
  N subjects: 97
  Mean: 0.6931
  Median: 0.7025
  SD: 0.1313
  SEM: 0.0133
  Min: 0.3306
  Max: 0.9848
  Range: 0.6542

Percentiles:
  10th: 0.5404
  25th: 0.6077
  50th: 0.7025
  75th: 0.7812
  90th: 0.8461

Performance Categories:
  Below chance (<50%): 5 (5.2%)
  Below 60%: 22 (22.7%)
  Above 80%: 19 (19.6%)
  Above 90%: 6 (6.2%)


In [14]:
# Identify lowest performers
print("\n" + "-"*80)
print("LOWEST PERFORMING SUBJECTS (Bottom 20)")
print("-"*80)

bottom_20 = overall_acc.nsmallest(20, 'accuracy')
print(bottom_20[['subject_id', 'accuracy']].to_string(index=False))

# Save for later analysis
bottom_20_ids = set(bottom_20['subject_id'])


--------------------------------------------------------------------------------
LOWEST PERFORMING SUBJECTS (Bottom 20)
--------------------------------------------------------------------------------
       subject_id  accuracy
0901_1300_9M4VCHG  0.330645
0828_1300_9M4VCHG  0.371681
0928_1600_539136F  0.372727
0816_1400_9M4VCHG  0.395161
0915_1000_539136F  0.436975
0823_1400_539136F  0.503817
0826_1300_U9TEJGM  0.512821
0817_1000_539136F  0.526718
0819_1400_539136F  0.534884
0927_0930_U9TEJGM  0.535354
0819_1400_9M4VCHG  0.543689
0901_1300_U9TEJGM  0.546875
0825_1000_9M4VCHG  0.555556
0813_1600_9M4VCHG  0.561538
0825_1300_539136F  0.564516
0813_1000_539136F  0.573643
0826_1000_9M4VCHG  0.574803
0901_1000_539136F  0.576000
0823_1400_U9TEJGM  0.577236
0831_1300_U9TEJGM  0.589147


In [15]:
# Highest performers
print("\n" + "-"*80)
print("HIGHEST PERFORMING SUBJECTS (Top 20)")
print("-"*80)

top_20 = overall_acc.nlargest(20, 'accuracy')
print(top_20[['subject_id', 'accuracy']].to_string(index=False))

top_20_ids = set(top_20['subject_id'])


--------------------------------------------------------------------------------
HIGHEST PERFORMING SUBJECTS (Top 20)
--------------------------------------------------------------------------------
       subject_id  accuracy
0823_1400_9M4VCHG  0.984848
0917_1030_9M4VCHG  0.968992
0923_1000_539136F  0.946565
0924_1000_9M4VCHG  0.915385
0930_1700_539136F  0.905263
1005_1600_U9TEJGM  0.901961
0830_1300_9M4VCHG  0.885496
0901_1300_539136F  0.870690
0924_1600_539136F  0.864407
0831_1300_539136F  0.853846
0831_1000_U9TEJGM  0.840909
0831_1300_9M4VCHG  0.840336
1005_1600_9M4VCHG  0.837398
0816_1400_539136F  0.829457
0922_1000_539136F  0.829268
0924_1000_U9TEJGM  0.816667
0818_1600_9M4VCHG  0.811111
0915_1000_9M4VCHG  0.809160
0826_1000_539136F  0.803030
0928_1600_U9TEJGM  0.795699


## 3. Cross-Cutting Analysis: Performance Across Conditions

In [16]:
print("\n" + "="*80)
print("CROSS-CUTTING ANALYSIS: PERFORMANCE ACROSS CONDITIONS")
print("="*80)

# Merge all data for comprehensive view
# Pivot ambiguity data
amb_pivot = ambiguity_acc.pivot_table(
    index='subject_id',
    columns='ambiguity_group',
    values='accuracy'
).reset_index()
amb_pivot.columns.name = None
# Columns come out alphabetically: High, Low, Medium - rename accordingly
amb_pivot = amb_pivot.rename(columns={'High': 'amb_high', 'Low': 'amb_low', 'Medium': 'amb_medium'})

# Pivot RT data
rt_pivot = rt_acc.pivot_table(
    index='subject_id',
    columns='rt_group',
    values='accuracy'
).reset_index()
rt_pivot.columns.name = None
# Columns come out alphabetically: Fast, Slow - rename accordingly
rt_pivot = rt_pivot.rename(columns={'Fast': 'rt_fast', 'Slow': 'rt_slow'})

# Merge all together
comprehensive = overall_acc[['subject_id', 'accuracy']].copy()
comprehensive.columns = ['subject_id', 'overall_acc']
comprehensive = comprehensive.merge(amb_pivot, on='subject_id', how='left')
comprehensive = comprehensive.merge(rt_pivot, on='subject_id', how='left')

print(f"\nComprehensive dataset: {len(comprehensive)} subjects")
print(f"Columns: {comprehensive.columns.tolist()}")


CROSS-CUTTING ANALYSIS: PERFORMANCE ACROSS CONDITIONS


KeyError: 'subject_id'

In [ ]:
# Calculate subject-level statistics
comprehensive['mean_across_amb'] = comprehensive[['amb_low', 'amb_medium', 'amb_high']].mean(axis=1)
comprehensive['std_across_amb'] = comprehensive[['amb_low', 'amb_medium', 'amb_high']].std(axis=1)
comprehensive['range_across_amb'] = comprehensive[['amb_low', 'amb_medium', 'amb_high']].max(axis=1) - comprehensive[['amb_low', 'amb_medium', 'amb_high']].min(axis=1)

comprehensive['mean_across_rt'] = comprehensive[['rt_fast', 'rt_slow']].mean(axis=1)
comprehensive['std_across_rt'] = comprehensive[['rt_fast', 'rt_slow']].std(axis=1)

print("\nCalculated cross-condition statistics")

In [ ]:
# Identify consistently poor performers (low across ALL conditions)
consistently_poor = comprehensive[
    (comprehensive['overall_acc'] < 0.6) &
    (comprehensive['amb_low'] < 0.7) &
    (comprehensive['amb_medium'] < 0.7) &
    (comprehensive['amb_high'] < 0.6)
]

print("\n" + "-"*80)
print("CONSISTENTLY POOR PERFORMERS (Low across ALL conditions)")
print("-"*80)
print(f"\nCriteria: Overall<60%, Low Amb<70%, Medium Amb<70%, High Amb<60%")
print(f"\nFound {len(consistently_poor)} subjects:")
if len(consistently_poor) > 0:
    print(consistently_poor[['subject_id', 'overall_acc', 'amb_low', 'amb_medium', 'amb_high']].to_string(index=False))

In [ ]:
# Identify subjects who struggle specifically with high ambiguity
high_amb_strugglers = comprehensive[
    (comprehensive['amb_high'] < 0.5) &
    (comprehensive['amb_low'] > 0.6)
]

print("\n" + "-"*80)
print("HIGH AMBIGUITY STRUGGLERS (Low<50% in High Amb, but >60% in Low Amb)")
print("-"*80)
print(f"\nFound {len(high_amb_strugglers)} subjects:")
if len(high_amb_strugglers) > 0:
    print(high_amb_strugglers[['subject_id', 'overall_acc', 'amb_low', 'amb_medium', 'amb_high']].sort_values('amb_high').to_string(index=False))

In [ ]:
# Identify subjects with high variability across conditions
high_variability = comprehensive.nlargest(10, 'range_across_amb')

print("\n" + "-"*80)
print("HIGH VARIABILITY SUBJECTS (Top 10 by range across ambiguity)")
print("-"*80)
print("These subjects show inconsistent performance across different trial types")
print(high_variability[['subject_id', 'overall_acc', 'amb_low', 'amb_medium', 'amb_high', 'range_across_amb']].to_string(index=False))

## 4. Visualize Performance Patterns

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Overall accuracy distribution
ax1 = axes[0, 0]
ax1.hist(overall_acc['accuracy'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax1.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Chance')
ax1.axvline(x=overall_acc['accuracy'].mean(), color='green', linestyle='-', linewidth=2, label='Mean')
ax1.set_xlabel('Accuracy', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of Subjects', fontsize=11, fontweight='bold')
ax1.set_title('A. Overall Accuracy Distribution', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Accuracy by ambiguity group (violin plot)
ax2 = axes[0, 1]
colors_amb = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
sns.violinplot(data=ambiguity_acc, x='ambiguity_group', y='accuracy', 
               order=['Low', 'Medium', 'High'], palette=colors_amb, ax=ax2)
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Ambiguity Group', fontsize=11, fontweight='bold')
ax2.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
ax2.set_title('B. Accuracy by Ambiguity Group', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

# 3. Accuracy by RT group
ax3 = axes[0, 2]
colors_rt = {'Fast': '#3498db', 'Slow': '#9b59b6'}
sns.violinplot(data=rt_acc, x='rt_group', y='accuracy',
               order=['Fast', 'Slow'], palette=colors_rt, ax=ax3)
ax3.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax3.set_xlabel('Reaction Time Group', fontsize=11, fontweight='bold')
ax3.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
ax3.set_title('C. Accuracy by Reaction Time', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)

# 4. Low vs High ambiguity scatter
ax4 = axes[1, 0]
ax4.scatter(comprehensive['amb_low'], comprehensive['amb_high'], alpha=0.6, s=50, color='steelblue')
ax4.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='y=x')
ax4.axhline(y=0.5, color='red', linestyle=':', alpha=0.5)
ax4.axvline(x=0.5, color='red', linestyle=':', alpha=0.5)
ax4.set_xlabel('Low Ambiguity Accuracy', fontsize=11, fontweight='bold')
ax4.set_ylabel('High Ambiguity Accuracy', fontsize=11, fontweight='bold')
ax4.set_title(f'D. Low vs High Ambiguity\n(r={comprehensive["amb_low"].corr(comprehensive["amb_high"]):.3f})', 
              fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Variability analysis
ax5 = axes[1, 1]
ax5.scatter(comprehensive['overall_acc'], comprehensive['std_across_amb'], alpha=0.6, s=50, color='purple')
ax5.set_xlabel('Overall Accuracy', fontsize=11, fontweight='bold')
ax5.set_ylabel('SD Across Ambiguity Groups', fontsize=11, fontweight='bold')
ax5.set_title('E. Consistency vs Performance', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3)

# 6. Cumulative distribution
ax6 = axes[1, 2]
sorted_acc = np.sort(overall_acc['accuracy'])
cumulative = np.arange(1, len(sorted_acc) + 1) / len(sorted_acc)
ax6.plot(sorted_acc, cumulative, linewidth=2, color='steelblue')
ax6.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Chance')
ax6.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax6.set_xlabel('Accuracy', fontsize=11, fontweight='bold')
ax6.set_ylabel('Cumulative Proportion', fontsize=11, fontweight='bold')
ax6.set_title('F. Cumulative Distribution', fontsize=12, fontweight='bold')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'comprehensive_subject_accuracy_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved")

## 5. Device/Session Pattern Analysis

In [ ]:
print("\n" + "="*80)
print("DEVICE/SESSION PATTERN ANALYSIS")
print("="*80)

# Extract device code from subject_id (last part after underscore)
comprehensive['device'] = comprehensive['subject_id'].apply(lambda x: x.split('_')[-1])
comprehensive['date'] = comprehensive['subject_id'].apply(lambda x: x.split('_')[0])
comprehensive['time'] = comprehensive['subject_id'].apply(lambda x: x.split('_')[1])

# Group by device
device_stats = comprehensive.groupby('device')['overall_acc'].agg([
    'count', 'mean', 'std', 'min', 'max'
]).round(4).sort_values('mean')

print("\nAccuracy by Device Code:")
print(device_stats.to_string())

# Identify problematic devices
print("\n" + "-"*80)
print("DEVICES WITH CONSISTENTLY LOW PERFORMANCE")
print("-"*80)

problematic_devices = device_stats[device_stats['mean'] < 0.65]
if len(problematic_devices) > 0:
    print(f"\nFound {len(problematic_devices)} devices with mean accuracy < 0.65:")
    print(problematic_devices.to_string())
    
    # List subjects from these devices
    for device in problematic_devices.index:
        device_subjects = comprehensive[comprehensive['device'] == device]
        print(f"\n  Device {device} subjects ({len(device_subjects)}):")
        print(f"  {device_subjects['subject_id'].tolist()}")

## 6. Summary and Recommendations

In [ ]:
print("\n" + "="*80)
print("SUMMARY AND RECOMMENDATIONS")
print("="*80)

# Calculate impact of removing poor performers
current_mean = comprehensive['overall_acc'].mean()

for threshold in [0.40, 0.45, 0.50, 0.55]:
    filtered = comprehensive[comprehensive['overall_acc'] >= threshold]
    n_removed = len(comprehensive) - len(filtered)
    new_mean = filtered['overall_acc'].mean()
    improvement = new_mean - current_mean
    
    print(f"\nRemoving subjects with overall accuracy < {threshold:.0%}:")
    print(f"  Subjects removed: {n_removed} ({100*n_removed/len(comprehensive):.1f}%)")
    print(f"  New mean: {new_mean:.4f} (was {current_mean:.4f})")
    print(f"  Improvement: +{improvement:.4f} ({100*improvement/current_mean:+.2f}%)")

# Key findings summary
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

print(f"""
1. OVERALL PERFORMANCE:
   - Mean accuracy: {current_mean:.4f}
   - {len(below_chance)} subjects ({100*len(below_chance)/len(overall_acc):.1f}%) perform BELOW CHANCE
   - {len(above_90)} subjects ({100*len(above_90)/len(overall_acc):.1f}%) achieve >90% accuracy

2. CONSISTENTLY POOR PERFORMERS:
   - {len(consistently_poor)} subjects perform poorly across ALL conditions
   - These are candidates for data quality investigation

3. CONDITION-SPECIFIC STRUGGLES:
   - {len(high_amb_strugglers)} subjects struggle specifically with high ambiguity
   - Low ambiguity mean: {amb_low['accuracy'].mean():.4f}
   - High ambiguity mean: {amb_high['accuracy'].mean():.4f}

4. DEVICE PATTERNS:
   - {len(problematic_devices)} device codes show consistently low performance
   - May indicate hardware/software issues

5. RECOMMENDATIONS:
   a) Investigate the {len(below_chance)} below-chance subjects for data quality
   b) Check eye-tracking quality for problematic device codes
   c) Consider trial-level analysis for consistently poor performers
   d) Examine whether excluding {len(below_chance)} subjects improves model generalization
""")

# Save comprehensive results
comprehensive.to_csv(OUTPUT_DIR / 'comprehensive_subject_accuracy_full.csv', index=False)

# Save summaries
summary_dict = {
    'below_chance_subjects': below_chance['subject_id'].tolist(),
    'consistently_poor_subjects': consistently_poor['subject_id'].tolist() if len(consistently_poor) > 0 else [],
    'high_amb_strugglers': high_amb_strugglers['subject_id'].tolist() if len(high_amb_strugglers) > 0 else [],
    'top_20_subjects': top_20['subject_id'].tolist(),
    'bottom_20_subjects': bottom_20['subject_id'].tolist(),
    'problematic_devices': problematic_devices.index.tolist() if len(problematic_devices) > 0 else [],
}

import json
with open(OUTPUT_DIR / 'subject_accuracy_summary.json', 'w') as f:
    json.dump(summary_dict, f, indent=2)

print(f"\nResults saved to {OUTPUT_DIR}/")
print("  - comprehensive_subject_accuracy_full.csv")
print("  - subject_accuracy_summary.json")
print("  - comprehensive_subject_accuracy_analysis.png")